# Adaptive Low-Latency Fraud Detection in Streaming Financial Systems with LLM-Augmented Explainability

## Notebook 04: Objective 1 (E1 & E2) — Adaptation Policy Effectiveness on Chronological Financial Transaction Stream

**Document Status:** Production Research Experiment (Milestone: Objective 1)  
**Primary Dataset:** IEEE-CIS Fraud Detection (`train_transaction.csv` + `train_identity.csv`)    
**Execution Scope:** 4 Policies (P0, P1, P2, P3) × 10 Paired Seeds = **40 Prequential Streaming Runs**  

### Research Questions & Hypotheses

- **Research Question 1 (RQ1):** *How does the choice of model adaptation policy affect fraud-detection performance and operational cost under concept drift in streaming financial transaction data?*
- **Hypothesis 1 (H1):** *Adaptive model-update policies (P1, P2, P3) achieve higher overall prequential PR-AUC on non-stationary financial transaction streams compared with an incremental-only baseline (P0), while incurring measurable operational cost trade-offs.*

### Scientific Invariants & Experimental Architecture (VERDICT A)

1. **Natural Stream Evaluation:** E1 and E2 execute on the unmodified chronological stream of IEEE-CIS (590,540 transactions, sorted strictly by `TransactionDT`).
2. **Primary Inferential Metric:** Full-stream prequential **PR-AUC / Average Precision** across all 501,959 streaming observations.
3. **Statistical Inferential Unit:** One full-stream PR-AUC observation per random seed ($N = 10$ paired seeds across policies).
4. **Avoidance of Pseudo-Replication:** Rolling-window PR-AUC trajectories provide visual/longitudinal diagnostic evidence; rolling windows are **strictly not pooled** as independent statistical observations.
5. **Confound Elimination:** No policy-dependent detector alarms (e.g. P2 ADWIN triggers) are used as shared evaluation boundaries. Discrete recovery time ($T_{\text{recovery}}$) and post-drift degradation slopes are formally reserved for controlled synthetic drift experiments (E3/E4).
6. **Falsifiability:** Hypothesis H1 is evaluated empirically against the evidence. Statistical significance is an empirical finding, **not** an experimental completion prerequisite; P0 superiority or non-significant differences are valid scientific outcomes.

## 1. Import Required Libraries and Environment Setup

We import standard numerical, data processing, statistical, and visualization libraries. In Kaggle environments, missing streaming dependencies (`river`, `scikit-learn`, `psutil`) are automatically installed.

In [1]:
from pathlib import Path
import sys
import os
import gc
import json
import time
import zipfile
import shutil
import warnings

# Automatic dependency bootstrap for cloud/Kaggle environments
# Uses --no-deps to prevent pip from modifying pre-installed numpy/scipy in the running container
try:
    import river
except ImportError:
    print("[ENVIRONMENT] river not found. Installing river and narwhals via pip (--no-deps)...")
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "river>=0.21.0", "narwhals>=2.0.0",
        "--no-deps", "-q"
    ])
    import river
    print(f"[ENVIRONMENT] river successfully installed: version {river.__version__}")

import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python Environment: Python {sys.version.split()[0]} on {sys.platform}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | SciPy: {scipy.__version__}")

Python Environment: Python 3.12.4 on win32
NumPy: 2.5.1 | Pandas: 3.0.3 | SciPy: 1.18.1


## 2. Execution Mode Configuration

To support both rapid local engine validation and full-scale empirical research on cloud infrastructure, execution mode is controlled via a single top-level flag:
- `'LOCAL_CPU'` : Fast smoke execution on a 3,000-row slice with 2 seeds to validate all pipelines, checkpointing, and statistical exports locally.
- `'KAGGLE'`    : Full-scale research execution across all 590,540 IEEE-CIS transactions across all 10 paired seeds (40 runs).

In [2]:
# TOP-LEVEL EXECUTION MODE SELECTION
# Options:
#   - 'LOCAL_CPU' : Fast smoke execution (validates pipeline, checkpointing, stats, plots)
#   - 'KAGGLE'    : Full 40-run research execution on complete IEEE-CIS dataset

RUN_MODE = 'LOCAL_CPU'  # Change to 'KAGGLE' when executing on Kaggle GPU/CPU instance

if RUN_MODE == 'LOCAL_CPU':
    PILOT_ROWS = 3000           # Fast local slice for end-to-end pipeline validation
    WARMUP_RATIO = 0.15         # 15% warmup
    ACTIVE_SEEDS = [42, 101]    # 2 seeds for local smoke testing of paired statistics
    VERBOSE = True
    print(f"[RUN_MODE] Running in LOCAL_CPU smoke mode ({PILOT_ROWS:,} rows, 2 seeds).")
elif RUN_MODE == 'KAGGLE':
    PILOT_ROWS = None           # Full 590,540 IEEE-CIS transactions
    WARMUP_RATIO = 0.15         # Exact 15% warmup = 88,581 transactions
    ACTIVE_SEEDS = [42, 101, 202, 303, 404, 505, 606, 707, 808, 909]  # Exact 10 frozen seeds
    VERBOSE = True
    print(f"[RUN_MODE] Running in KAGGLE research mode (Full 590k stream, all 10 seeds, 40 runs).")
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}. Expected 'LOCAL_CPU' or 'KAGGLE'.")

[RUN_MODE] Running in LOCAL_CPU smoke mode (3,000 rows, 2 seeds).


## 3. Project Root & Directory Hierarchy Configuration

We dynamically locate the repository root via `pyproject.toml` and configure the output directory hierarchy for tables, figures, checkpoints, and export bundles.

In [3]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for parent in [current] + list(current.parents):
        if (parent / 'pyproject.toml').exists() and (parent / 'src').exists():
            return parent
    return None

if RUN_MODE == 'LOCAL_CPU':
    PROJECT_ROOT = find_project_root(Path.cwd())
    if PROJECT_ROOT is None:
        PROJECT_ROOT = Path('.').resolve()
    print(f"[LOCAL] Repository root resolved: {PROJECT_ROOT}")
elif RUN_MODE == 'KAGGLE':
    KAGGLE_WORKING = Path('/kaggle/working')
    REPO_NAME = 'adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability'
    PROJECT_ROOT = KAGGLE_WORKING / REPO_NAME
    REPO_URL = 'https://github.com/ParminderSinghGithub/adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability.git'

    if not (PROJECT_ROOT / 'src').exists():
        print(f"[KAGGLE] Cloning repository from {REPO_URL} into {PROJECT_ROOT}...")
        os.system(f"git clone {REPO_URL} {PROJECT_ROOT}")
    else:
        print(f"[KAGGLE] Pulling latest repository updates in {PROJECT_ROOT}...")
        os.system(f"cd {PROJECT_ROOT} && git pull origin main")

    if not (PROJECT_ROOT / 'src').exists() and (KAGGLE_WORKING / 'src').exists():
        PROJECT_ROOT = KAGGLE_WORKING

    print(f"[KAGGLE] Repository root resolved: {PROJECT_ROOT}")

# Anchor Python module resolution strictly to repository root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path.")

assert (PROJECT_ROOT / 'src').exists(), f"Source directory 'src' not found in {PROJECT_ROOT}"

# Output hierarchy
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
RUN_CHECKPOINT_DIR = CHECKPOINT_DIR / 'objective1_runs'
KAGGLE_ARTIFACT_DIR = PROJECT_ROOT / 'kaggle_artifacts'

for d in [TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR, RUN_CHECKPOINT_DIR, KAGGLE_ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def save_table(df: pd.DataFrame, filename: str) -> Path:
    target = TABLE_DIR / f"{filename}.csv"
    df.to_csv(target, index=False)
    print(f"  [TABLE SAVED] {target.name} ({len(df)} rows)")
    return target

def save_figure(fig: plt.Figure, filename: str) -> Path:
    target = FIGURE_DIR / f"{filename}.png"
    fig.savefig(target, dpi=300, bbox_inches='tight')
    print(f"  [FIGURE SAVED] {target.name}")
    return target

[LOCAL] Repository root resolved: C:\Projects\Thesis
Added C:\Projects\Thesis to sys.path.


## 4. Benchmark Dataset Discovery & Path Resolution

We resolve the location of the IEEE-CIS transaction and identity files, searching the local workspace and standard Kaggle input directories (`/kaggle/input/ieee-fraud-detection`).

In [4]:
def resolve_ieee_cis_dir() -> Path:
    # 1. Direct candidate directories
    candidates = [
        PROJECT_ROOT / 'datasets' / 'ieee_cis',
        PROJECT_ROOT / 'data' / 'raw' / 'ieee-cis-fraud-detection',
        Path('/kaggle/input/ieee-fraud-detection'),
        Path('/kaggle/input/ieee-cis-fraud-detection'),
        Path('/kaggle/working/data/ieee_cis')
    ]
    for c in candidates:
        if c.exists() and (c / 'train_transaction.csv').exists():
            return c

    # 2. Dynamic recursive discovery under /kaggle/input (cloud execution)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob('train_transaction.csv'))
        if matches:
            return matches[0].parent

        # Check for zip files to auto-extract
        zip_matches = (
            list(kaggle_input.rglob('*ieee*.zip')) +
            list(kaggle_input.rglob('*fraud*.zip')) +
            list(kaggle_input.rglob('*transaction*.zip'))
        )
        if zip_matches:
            target_extract = Path('/kaggle/working/data/ieee_cis')
            target_extract.mkdir(parents=True, exist_ok=True)
            print(f"[KAGGLE] Auto-extracting {zip_matches[0].name} to {target_extract}...")
            with zipfile.ZipFile(zip_matches[0], 'r') as zf:
                zf.extractall(target_extract)
            extracted_matches = list(target_extract.rglob('train_transaction.csv'))
            if extracted_matches:
                return extracted_matches[0].parent
            return target_extract

    # 3. Dynamic recursive discovery under local workspace
    local_matches = list(PROJECT_ROOT.rglob('train_transaction.csv'))
    if local_matches:
        return local_matches[0].parent

    raise FileNotFoundError(
        "Could not locate IEEE-CIS dataset ('train_transaction.csv'). "
        f"Searched candidates: {candidates} and recursively under {kaggle_input if kaggle_input.exists() else PROJECT_ROOT}"
    )

IEEE_CIS_DIR = resolve_ieee_cis_dir()
print(f"IEEE-CIS data source resolved at: {IEEE_CIS_DIR}")

IEEE-CIS data source resolved at: C:\Projects\Thesis\datasets\ieee_cis


## 5. Environment & Core Dependency Verification

We verify that all modules and libraries required for prequential streaming execution, concept drift monitoring, and statistical inference are loaded and operational.

In [5]:
from src.utils.seed import set_seed
from src.data.loader import load_ieee_cis
from src.data.preprocessing import StreamingPreprocessor
from src.models.base_learner import HoeffdingTreeLearner
from src.adaptation.retrainer import RetrainingEngine
from src.drift.monitor import DriftMonitor
from src.adaptation.policy import PolicyManager
from src.evaluation.metrics import StreamingMetricsTracker
from src.pipeline.runner import PrequentialRunner, RunResult

dependency_manifest = [
    {"Component": "River Streaming ML", "Status": "PASS"},
    {"Component": "Scikit-Learn Evaluation", "Status": "PASS"},
    {"Component": "SciPy Statistical Tests", "Status": "PASS"},
    {"Component": "Pipeline Runner Module", "Status": "PASS"},
    {"Component": "Streaming Preprocessor", "Status": "PASS"},
    {"Component": "Hoeffding Tree Learner", "Status": "PASS"},
    {"Component": "ADWIN Drift Monitor", "Status": "PASS"},
    {"Component": "Retraining Engine", "Status": "PASS"},
    {"Component": "Adaptation Policy Engine", "Status": "PASS"}
]
dep_df = pd.DataFrame(dependency_manifest)
assert all(r["Status"] == "PASS" for r in dependency_manifest), "Critical dependency missing."
print("All core dependencies verified successfully.")

All core dependencies verified successfully.


## 6. Frozen Objective 1 Experimental Configuration & Invariant Asserts

We declare the frozen experimental parameters established in the Master Source of Truth (§6, §8, §13, §19) and assert their exact values prior to execution.

In [6]:
# FROZEN OBJECTIVE 1 CONFIGURATION (Source of Truth §6, §8, §13, §19)
FROZEN_CONFIG = {
    "dataset": "ieee_cis",
    "total_rows": 590540,
    "warmup_ratio": 0.15,
    "warmup_rows": 88581,
    "stream_rows": 501959,
    "policies": ["P0", "P1", "P2", "P3"],
    "seeds": [42, 101, 202, 303, 404, 505, 606, 707, 808, 909],
    "base_learner": "HoeffdingTreeClassifier",
    "grace_period": 200,
    "learner_delta": 1e-7,
    "drift_detector": "ADWIN",
    "adwin_delta": 0.002,
    "p1_interval": 10000,
    "retraining_window": 5000,
    "p3_segment_col": "ProductCD",
    "p3_min_segment_size": 500,
    "p0_mode": "incremental"
}

# Runtime parameters conditioned on execution mode
if RUN_MODE == 'KAGGLE':
    TARGET_WARMUP = FROZEN_CONFIG["warmup_rows"]
    TARGET_SEEDS = FROZEN_CONFIG["seeds"]
    P1_INTERVAL = FROZEN_CONFIG["p1_interval"]
    WINDOW_SIZE = FROZEN_CONFIG["retraining_window"]
else:
    TARGET_WARMUP = int(PILOT_ROWS * FROZEN_CONFIG["warmup_ratio"])
    TARGET_SEEDS = ACTIVE_SEEDS
    P1_INTERVAL = 1000  # Scaled for local pilot validation
    WINDOW_SIZE = 500   # Scaled for local pilot validation

ADWIN_DELTA = FROZEN_CONFIG["adwin_delta"]
GRACE_PERIOD = FROZEN_CONFIG["grace_period"]
LEARNER_DELTA = FROZEN_CONFIG["learner_delta"]
P3_MIN_SEG = FROZEN_CONFIG["p3_min_segment_size"]
POLICIES = FROZEN_CONFIG["policies"]

print("FROZEN OBJECTIVE 1 EXPERIMENT SPECIFICATION:")
print(f"  Target Policies:         {POLICIES}")
print(f"  Target Seeds:            {TARGET_SEEDS} (Total: {len(TARGET_SEEDS)} seeds)")
print(f"  Total Experimental Runs: {len(POLICIES) * len(TARGET_SEEDS)}")
print(f"  P1 Retrain Interval:     {P1_INTERVAL:,} tx")
print(f"  Retraining Window (W):   {WINDOW_SIZE:,} tx")
print(f"  ADWIN Sensitivity:       delta = {ADWIN_DELTA}")
print(f"  Hoeffding Tree:          grace_period = {GRACE_PERIOD}, delta = {LEARNER_DELTA}")
print(f"  P3 Segmentation:         ProductCD (min_segment = {P3_MIN_SEG})")

FROZEN OBJECTIVE 1 EXPERIMENT SPECIFICATION:
  Target Policies:         ['P0', 'P1', 'P2', 'P3']
  Target Seeds:            [42, 101] (Total: 2 seeds)
  Total Experimental Runs: 8
  P1 Retrain Interval:     1,000 tx
  Retraining Window (W):   500 tx
  ADWIN Sensitivity:       delta = 0.002
  Hoeffding Tree:          grace_period = 200, delta = 1e-07
  P3 Segmentation:         ProductCD (min_segment = 500)


## 7. IEEE-CIS Dataset Ingestion & Chronological Monotonicity Verification

We load the transaction table and left-join the identity attributes on `TransactionID`. Chronological stream monotonicity is strictly validated on `TransactionDT`.

In [7]:
print(f"Loading IEEE-CIS transaction and identity data (RUN_MODE={RUN_MODE})...")
t0_load = time.time()

if PILOT_ROWS is not None:
    tx_file = IEEE_CIS_DIR / 'train_transaction.csv'
    id_file = IEEE_CIS_DIR / 'train_identity.csv'
    df_tx = pd.read_csv(tx_file, nrows=PILOT_ROWS)
    if id_file.exists():
        df_id = pd.read_csv(id_file)
        df_raw = pd.merge(df_tx, df_id, on='TransactionID', how='left')
    else:
        df_raw = df_tx
else:
    df_raw = load_ieee_cis(data_dir=IEEE_CIS_DIR, split='train', join_identity=True)

load_time = time.time() - t0_load
print(f"Dataset ingested in {load_time:.2f}s. Loaded shape: {df_raw.shape}")

# Verify expected dimensions
if RUN_MODE == 'KAGGLE':
    assert len(df_raw) == FROZEN_CONFIG["total_rows"], f"Row count mismatch: expected {FROZEN_CONFIG['total_rows']}, got {len(df_raw)}"

# Strict chronological monotonicity check
assert 'TransactionDT' in df_raw.columns, "Missing TransactionDT temporal column."
if not df_raw['TransactionDT'].is_monotonic_increasing:
    print("Enforcing strict chronological ordering by TransactionDT...")
    df_raw = df_raw.sort_values('TransactionDT').reset_index(drop=True)

assert df_raw['TransactionDT'].is_monotonic_increasing, "Temporal monotonicity violation."
assert 'isFraud' in df_raw.columns, "Missing target isFraud."
assert 'ProductCD' in df_raw.columns, "Missing ProductCD segment column."

print(f"Chronological stream ordering verified: {len(df_raw):,} transactions sorted by TransactionDT.")
fraud_pct = df_raw['isFraud'].mean() * 100
print(f"Target distribution (isFraud): {df_raw['isFraud'].sum():,} frauds ({fraud_pct:.3f}% prevalence)")
print("Segment distribution (ProductCD):")
print(df_raw['ProductCD'].value_counts().to_dict())

Loading IEEE-CIS transaction and identity data (RUN_MODE=LOCAL_CPU)...
Dataset ingested in 1.65s. Loaded shape: (3000, 434)
Chronological stream ordering verified: 3,000 transactions sorted by TransactionDT.
Target distribution (isFraud): 59 frauds (1.967% prevalence)
Segment distribution (ProductCD):
{'W': 2370, 'C': 241, 'H': 223, 'R': 90, 'S': 76}


## 8. Warmup Split & Leakage-Safe Streaming Preprocessing

We allocate the first 15.0% of the stream strictly to historical warmup ($N_{\text{warmup}} = 88{,}581$ in full mode). The `StreamingPreprocessor` is fitted exclusively on $df_{\text{warmup}}$, excluding target, temporal, and ID columns.

In [8]:
n_total = len(df_raw)
actual_warmup = min(TARGET_WARMUP, int(n_total * 0.40))
actual_stream = n_total - actual_warmup

df_warmup = df_raw.iloc[:actual_warmup].copy()
df_stream = df_raw.iloc[actual_warmup:].copy()

print(f"Warmup allocation:    {len(df_warmup):,} transactions (train-only, strictly un-evaluated)")
print(f"Streaming evaluation: {len(df_stream):,} transactions (strict prequential test-then-train)")

# Initialize and fit StreamingPreprocessor strictly on warmup
preprocessor = StreamingPreprocessor(
    dataset_name='ieee_cis',
    scale_features=False,
    custom_categorical_cols=['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']
)
t0_fit = time.time()
preprocessor.fit(df_warmup)
fit_time = time.time() - t0_fit
print(f"Preprocessor fitted strictly on warmup slice in {fit_time:.2f}s.")

# Transform full stream using warmup-derived parameters
t0_trans = time.time()
X_all, y_all, seg_all = preprocessor.transform(df_raw)
trans_time = time.time() - t0_trans
print(f"Stream transformed in {trans_time:.2f}s. Processed feature matrix shape: {X_all.shape}")

# Prequential causal leakage assertions
assert len(X_all) == n_total, "Row count mismatch in transformed X."
assert len(y_all) == n_total, "Row count mismatch in target y."
assert len(seg_all) == n_total, "Row count mismatch in segment seg."
assert 'TransactionDT' not in X_all.columns, "Leakage violation: TransactionDT present in features."
assert 'isFraud' not in X_all.columns, "Leakage violation: isFraud target present in features."
assert 'TransactionID' not in X_all.columns, "Leakage violation: TransactionID present in features."
assert X_all.isna().sum().sum() == 0, "Unimputed missing values detected in feature matrix."

print("[PASS] Preprocessing integrity verified: zero temporal leakage, zero feature NaNs.")

Warmup allocation:    450 transactions (train-only, strictly un-evaluated)
Streaming evaluation: 2,550 transactions (strict prequential test-then-train)
Preprocessor fitted strictly on warmup slice in 0.42s.


C:\Projects\Thesis\src\data\preprocessing.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_df[col] = series
C:\Projects\Thesis\src\data\preprocessing.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_df[col] = series
C:\Projects\Thesis\src\data\preprocessing.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `ne

Stream transformed in 0.52s. Processed feature matrix shape: (3000, 431)
[PASS] Preprocessing integrity verified: zero temporal leakage, zero feature NaNs.


## 9. Statistical Hypothesis Testing & Effect Size Framework

We implement the locked statistical testing protocol (Source of Truth §13):
1. Paired difference test across 10 random seeds (Paired Student's t-test if normality holds via Shapiro-Wilk; Wilcoxon signed-rank test otherwise).
2. **Holm-Šidák step-down procedure** for Family-Wise Error Rate (FWER) control.
3. Both **paired Cohen's $d_z$** and sample-size-corrected **paired Hedges' $g_z$** effect sizes.

In [9]:
def compute_paired_comparison(values_a, values_b, name_a="Adaptive", name_b="Baseline"):
    d = np.array(values_a) - np.array(values_b)
    n = len(d)
    mean_d = float(np.mean(d))
    median_d = float(np.median(d))
    std_d = float(np.std(d, ddof=1)) if n > 1 else 0.0

    # Normality test on paired differences
    if n >= 3:
        shapiro_stat, shapiro_p = stats.shapiro(d)
        shapiro_p = float(shapiro_p)
    else:
        shapiro_p = 1.0

    # Paired hypothesis test selection
    if std_d == 0.0:
        test_type = "identical_pairs"
        raw_p = 1.0
    elif shapiro_p >= 0.05:
        test_type = "paired_ttest"
        t_stat, raw_p = stats.ttest_rel(values_a, values_b)
    else:
        test_type = "wilcoxon"
        try:
            w_stat, raw_p = stats.wilcoxon(values_a, values_b)
        except Exception:
            t_stat, raw_p = stats.ttest_rel(values_a, values_b)
            test_type = "paired_ttest_fallback"

    raw_p = float(raw_p) if not np.isnan(raw_p) else 1.0

    # Paired Cohen's d_z = mean(D) / std(D)
    dz = float(mean_d / std_d) if std_d > 0 else 0.0

    # Sample-size corrected paired Hedges' g_z = d_z * (1 - 3 / (4*df - 1))
    df_n = n - 1
    j_factor = float(1.0 - (3.0 / (4.0 * df_n - 1.0))) if df_n > 0 else 1.0
    gz = float(dz * j_factor)

    direction = f"{name_a} > {name_b}" if mean_d > 0 else (f"{name_b} > {name_a}" if mean_d < 0 else "Identical")

    return {
        "n_pairs": n,
        "mean_diff": round(mean_d, 6),
        "median_diff": round(median_d, 6),
        "std_diff": round(std_d, 6),
        "shapiro_p": round(shapiro_p, 4),
        "test_type": test_type,
        "raw_p_value": raw_p,
        "cohens_dz": round(dz, 4),
        "hedges_gz": round(gz, 4),
        "direction": direction
    }

def apply_holm_sidak_correction(comparison_dict):
    m = len(comparison_dict)
    sorted_items = sorted(comparison_dict.items(), key=lambda item: item[1]["raw_p_value"])
    
    running_max = 0.0
    adjusted_dict = {}
    for rank, (key, res) in enumerate(sorted_items):
        step_factor = m - rank
        p_raw = res["raw_p_value"]
        p_sidak = 1.0 - (1.0 - p_raw) ** step_factor
        adj_p = max(running_max, min(1.0, p_sidak))
        running_max = adj_p
        
        row_copy = dict(res)
        row_copy["holm_sidak_p"] = round(adj_p, 6)
        row_copy["significant_005"] = bool(adj_p < 0.05)
        adjusted_dict[key] = row_copy
        
    return {k: adjusted_dict[k] for k in comparison_dict.keys()}

print("Statistical testing and effect size framework initialized.")

Statistical testing and effect size framework initialized.


## 10. Prequential Grid Orchestration with Checkpointing & Resumption

We implement `execute_single_run` to orchestrate an individual (Policy, Seed) prequential run. To protect against cloud disconnections, each completed run is checkpointed to `outputs/checkpoints/objective1_runs/run_{policy}_seed{seed}.json`. If a checkpoint already exists, it is loaded immediately without recomputation.

In [10]:
def compute_rolling_trajectories(records, window_size=5000, max_points=100):
    n_records = len(records)
    if n_records < window_size:
        window_size = max(50, n_records // 2)
    step = max(1, (n_records - window_size) // max_points)
    
    y_true_list = [r.y_true for r in records]
    y_prob_list = [r.y_prob for r in records]
    
    trajectory = []
    for start_idx in range(0, n_records - window_size + 1, step):
        end_idx = start_idx + window_size
        w_true = y_true_list[start_idx:end_idx]
        w_prob = y_prob_list[start_idx:end_idx]
        if len(set(w_true)) >= 2:
            pr_val = float(average_precision_score(w_true, w_prob))
        else:
            pr_val = 0.0
        trajectory.append({
            "stream_tx": end_idx,
            "rolling_pr_auc": round(pr_val, 5)
        })
    return trajectory

def execute_single_run(policy_name, seed, X_data, y_data, seg_data, warmup_size):
    ckpt_file = RUN_CHECKPOINT_DIR / f"run_{policy_name}_seed{seed}.json"
    
    # Checkpoint resumption
    if ckpt_file.exists():
        try:
            with open(ckpt_file, 'r', encoding='utf-8') as f:
                saved = json.load(f)
            print(f"  [RESUME] Loaded checkpoint: {policy_name} (Seed {seed}) -> PR-AUC: {saved['pr_auc']:.4f}")
            return saved
        except Exception as e:
            print(f"  [WARN] Failed loading {ckpt_file.name}, recomputing: {e}")

    # Set deterministic random seed
    set_seed(seed)
    
    # Construct runner with frozen production parameters
    runner = PrequentialRunner.from_config(
        policy_str=policy_name,
        detector_str='adwin',
        grace_period=GRACE_PERIOD,
        delta=LEARNER_DELTA,
        window_size=WINDOW_SIZE,
        n_interval=P1_INTERVAL,
        detector_kwargs={"delta": ADWIN_DELTA},
        segment_aware=(policy_name == 'P3'),
        seed=seed,
        p0_mode=FROZEN_CONFIG["p0_mode"]
    )
    
    t0_run = time.time()
    res = runner.run(
        X=X_data,
        y=y_data,
        segment=seg_data,
        warmup_size=warmup_size
    )
    run_duration = time.time() - t0_run
    
    # Extract point-in-time and operational metrics
    summary = res.summary_dict()
    lat = res.latency_percentiles
    
    # Compute compact rolling trajectory
    trajectory = compute_rolling_trajectories(res.records, window_size=WINDOW_SIZE, max_points=100)
    
    # Sum adaptation duration
    total_adapt_duration = sum(ev.get("duration_s", 0.0) for ev in res.adaptation_log)
    
    run_record = {
        "run_id": f"{policy_name}_seed{seed}",
        "policy": policy_name,
        "seed": seed,
        "warmup_tx": res.n_warmup,
        "stream_tx": res.n_stream,
        "adaptation_count": res.n_adaptations,
        "total_adaptation_duration_s": round(total_adapt_duration, 4),
        "drift_events": res.n_drift_events,
        "wall_time_s": round(res.total_wall_time_s, 2),
        "inference_p50_ms": round(lat.get("p50", 0.0) * 1e3, 3),
        "inference_p95_ms": round(lat.get("p95", 0.0) * 1e3, 3),
        "inference_p99_ms": round(lat.get("p99", 0.0) * 1e3, 3),
        "inference_mean_ms": round(lat.get("mean", 0.0) * 1e3, 3),
        "pr_auc": round(summary.get("pr_auc") or 0.0, 6),
        "roc_auc": round(summary.get("roc_auc") or 0.0, 6),
        "precision": round(summary.get("precision") or 0.0, 6),
        "recall": round(summary.get("recall") or 0.0, 6),
        "f1": round(summary.get("f1") or 0.0, 6),
        "f2": round(summary.get("f2") or 0.0, 6),
        "rolling_trajectory": trajectory
    }
    
    # Save checkpoint to disk
    with open(ckpt_file, 'w', encoding='utf-8') as f:
        json.dump(run_record, f, indent=2)
        
    print(f"  [COMPLETED] {policy_name} (Seed {seed}) -> PR-AUC: {run_record['pr_auc']:.4f} | p95 Latency: {run_record['inference_p95_ms']:.3f}ms | Wall: {run_record['wall_time_s']:.1f}s")
    
    # Explicit garbage collection to maintain flat memory profile
    del res
    del runner
    gc.collect()
    
    return run_record

## 11. Execution Grid Loop (Policies × Paired Seeds)

We execute the full factorial grid across policies P0, P1, P2, P3 and all specified paired seeds.

In [11]:
print(f"Beginning Objective 1 execution loop: {len(POLICIES)} policies × {len(TARGET_SEEDS)} seeds...")
t0_grid = time.time()

all_run_results = []
for p_idx, policy in enumerate(POLICIES, 1):
    print(f"\n--- [{p_idx}/{len(POLICIES)}] Executing Policy: {policy} ---")
    for s_idx, seed in enumerate(TARGET_SEEDS, 1):
        print(f"  -> Policy {policy} | Seed {seed} ({s_idx}/{len(TARGET_SEEDS)})")
        run_data = execute_single_run(
            policy_name=policy,
            seed=seed,
            X_data=X_all,
            y_data=y_all,
            seg_data=seg_all,
            warmup_size=actual_warmup
        )
        all_run_results.append(run_data)

total_grid_time = time.time() - t0_grid
print(f"\n[COMPLETE] All {len(all_run_results)} experimental runs finished in {total_grid_time:.2f}s.")

Beginning Objective 1 execution loop: 4 policies × 2 seeds...

--- [1/4] Executing Policy: P0 ---
  -> Policy P0 | Seed 42 (1/2)
  [RESUME] Loaded checkpoint: P0 (Seed 42) -> PR-AUC: 0.0182
  -> Policy P0 | Seed 101 (2/2)
  [RESUME] Loaded checkpoint: P0 (Seed 101) -> PR-AUC: 0.0182

--- [2/4] Executing Policy: P1 ---
  -> Policy P1 | Seed 42 (1/2)
  [RESUME] Loaded checkpoint: P1 (Seed 42) -> PR-AUC: 0.0183
  -> Policy P1 | Seed 101 (2/2)
  [RESUME] Loaded checkpoint: P1 (Seed 101) -> PR-AUC: 0.0183

--- [3/4] Executing Policy: P2 ---
  -> Policy P2 | Seed 42 (1/2)
  [RESUME] Loaded checkpoint: P2 (Seed 42) -> PR-AUC: 0.0182
  -> Policy P2 | Seed 101 (2/2)
  [RESUME] Loaded checkpoint: P2 (Seed 101) -> PR-AUC: 0.0182

--- [4/4] Executing Policy: P3 ---
  -> Policy P3 | Seed 42 (1/2)
  [RESUME] Loaded checkpoint: P3 (Seed 42) -> PR-AUC: 0.0299
  -> Policy P3 | Seed 101 (2/2)
  [RESUME] Loaded checkpoint: P3 (Seed 101) -> PR-AUC: 0.0299

[COMPLETE] All 8 experimental runs finished in 0.

## 12. Run-Level Results Assembly & Export

We extract tabular metrics from all completed runs into a master DataFrame and export `objective1_e1_e2_run_results.csv`.

In [12]:
# Assemble tabular results (excluding trajectory arrays)
run_table_rows = []
for r in all_run_results:
    row = {k: v for k, v in r.items() if k != 'rolling_trajectory'}
    run_table_rows.append(row)

run_results_df = pd.DataFrame(run_table_rows)
save_table(run_results_df, 'objective1_e1_e2_run_results')
display(run_results_df.head(10))

  [TABLE SAVED] objective1_e1_e2_run_results.csv (8 rows)


,run_id,policy,seed,warmup_tx,stream_tx,adaptation_count,total_adaptation_duration_s,drift_events,wall_time_s,inference_p50_ms,inference_p95_ms,inference_p99_ms,inference_mean_ms,pr_auc,roc_auc,precision,recall,f1,f2
0,P0_seed42,P0,42,450,2550,0,0.0000,0,21.97,0.022,0.041,0.056,0.024,0.018210,0.456708,0.0,0.000000,0.000000,0.0
1,P0_seed101,P0,101,450,2550,0,0.0000,0,22.47,0.026,0.040,0.059,0.027,0.018210,0.456708,0.0,0.000000,0.000000,0.0
2,P1_seed42,P1,42,450,2550,2,1.2330,0,19.80,0.015,0.026,0.033,0.017,0.018303,0.462782,0.0,0.000000,0.000000,0.0
3,P1_seed101,P1,101,450,2550,2,1.2529,0,19.99,0.015,0.025,0.033,0.017,0.018303,0.462782,0.0,0.000000,0.000000,0.0
4,P2_seed42,P2,42,450,2550,0,0.0000,0,18.83,0.016,0.026,0.034,0.017,0.018210,0.456708,0.0,0.000000,0.000000,0.0
5,P2_seed101,P2,101,450,2550,0,0.0000,0,19.22,0.016,0.030,0.039,0.018,0.018210,0.456708,0.0,0.000000,0.000000,0.0
6,P3_seed42,P3,42,450,2550,0,0.0000,0,24.26,0.016,0.063,0.785,0.034,0.029942,0.529755,0.0,0.019231,0.035714,0.0
7,P3_seed101,P3,101,450,2550,0,0.0000,0,18.09,0.016,0.030,0.788,0.028,0.029942,0.529755,0.0,0.019231,0.035714,0.0


## 13. Experiment E1: Incremental-Only Baseline vs. Adaptive Policies

We perform the primary hypothesis test for Hypothesis H1:
- Primary comparisons: **P0 vs. P1**, **P0 vs. P2**, **P0 vs. P3**
- Paired statistical tests across the random seed replications
- **Holm-Šidák step-down correction** across the 3 comparisons
- Paired Cohen's $d_z$ and sample-size-corrected Hedges' $g_z$ effect sizes.

In [13]:
e1_comparisons = {}
p0_praucs = run_results_df[run_results_df['policy'] == 'P0'].sort_values('seed')['pr_auc'].tolist()

for adaptive_pol in ['P1', 'P2', 'P3']:
    adapt_praucs = run_results_df[run_results_df['policy'] == adaptive_pol].sort_values('seed')['pr_auc'].tolist()
    key = f"P0_vs_{adaptive_pol}"
    comp_res = compute_paired_comparison(
        values_a=adapt_praucs,
        values_b=p0_praucs,
        name_a=adaptive_pol,
        name_b="P0"
    )
    e1_comparisons[key] = comp_res

# Apply Holm-Sidak correction across E1 primary comparisons
e1_corrected = apply_holm_sidak_correction(e1_comparisons)

e1_rows = []
for comp_name, stats_dict in e1_corrected.items():
    row = {"Comparison": comp_name}
    row.update(stats_dict)
    e1_rows.append(row)

e1_df = pd.DataFrame(e1_rows)
save_table(e1_df, 'objective1_e1_pairwise_statistics')

print("EXPERIMENT E1: INCREMENTAL BASELINE (P0) vs. ADAPTIVE POLICIES (P1, P2, P3)")
display(e1_df)

  [TABLE SAVED] objective1_e1_pairwise_statistics.csv (3 rows)
EXPERIMENT E1: INCREMENTAL BASELINE (P0) vs. ADAPTIVE POLICIES (P1, P2, P3)


,Comparison,n_pairs,mean_diff,median_diff,std_diff,shapiro_p,test_type,raw_p_value,cohens_dz,hedges_gz,direction,holm_sidak_p,significant_005
0,P0_vs_P1,2,0.000093,0.000093,0.0,1.0,identical_pairs,1.0,0.0,0.0,P1 > P0,1.0,False
1,P0_vs_P2,2,0.000000,0.000000,0.0,1.0,identical_pairs,1.0,0.0,0.0,Identical,1.0,False
2,P0_vs_P3,2,0.011732,0.011732,0.0,1.0,identical_pairs,1.0,0.0,0.0,P3 > P0,1.0,False


## 14. Experiment E2: Full Factorial Policy Comparison Matrix

We evaluate the complete pairwise comparison matrix across all 4 policies (6 pairwise combinations) with Holm-Šidák multiple-testing correction.

In [14]:
import itertools

e2_comparisons = {}
policy_prauc_map = {
    pol: run_results_df[run_results_df['policy'] == pol].sort_values('seed')['pr_auc'].tolist()
    for pol in POLICIES
}

for pol_a, pol_b in itertools.combinations(POLICIES, 2):
    key = f"{pol_a}_vs_{pol_b}"
    comp_res = compute_paired_comparison(
        values_a=policy_prauc_map[pol_a],
        values_b=policy_prauc_map[pol_b],
        name_a=pol_a,
        name_b=pol_b
    )
    e2_comparisons[key] = comp_res

e2_corrected = apply_holm_sidak_correction(e2_comparisons)

e2_rows = []
for comp_name, stats_dict in e2_corrected.items():
    row = {"Comparison": comp_name}
    row.update(stats_dict)
    e2_rows.append(row)

e2_df = pd.DataFrame(e2_rows)
save_table(e2_df, 'objective1_e2_pairwise_statistics')

print("EXPERIMENT E2: FULL POLICY COMPARISON MATRIX (6 PAIRWISE COMPARISONS)")
display(e2_df)

  [TABLE SAVED] objective1_e2_pairwise_statistics.csv (6 rows)
EXPERIMENT E2: FULL POLICY COMPARISON MATRIX (6 PAIRWISE COMPARISONS)


,Comparison,n_pairs,mean_diff,median_diff,std_diff,shapiro_p,test_type,raw_p_value,cohens_dz,hedges_gz,direction,holm_sidak_p,significant_005
0,P0_vs_P1,2,-0.000093,-0.000093,0.0,1.0,identical_pairs,1.0,0.0,0.0,P1 > P0,1.0,False
1,P0_vs_P2,2,0.000000,0.000000,0.0,1.0,identical_pairs,1.0,0.0,0.0,Identical,1.0,False
2,P0_vs_P3,2,-0.011732,-0.011732,0.0,1.0,identical_pairs,1.0,0.0,0.0,P3 > P0,1.0,False
3,P1_vs_P2,2,0.000093,0.000093,0.0,1.0,identical_pairs,1.0,0.0,0.0,P1 > P2,1.0,False
4,P1_vs_P3,2,-0.011639,-0.011639,0.0,1.0,identical_pairs,1.0,0.0,0.0,P3 > P1,1.0,False
5,P2_vs_P3,2,-0.011732,-0.011732,0.0,1.0,identical_pairs,1.0,0.0,0.0,P3 > P2,1.0,False


## 15. Policy-Level Predictive & Operational Summaries

We compute aggregate statistics (mean, standard deviation, median) for predictive performance and operational resource costs across policies.

In [15]:
# Predictive summary
pred_summary = run_results_df.groupby('policy').agg(
    Mean_PR_AUC=('pr_auc', 'mean'),
    Std_PR_AUC=('pr_auc', 'std'),
    Median_PR_AUC=('pr_auc', 'median'),
    Mean_ROC_AUC=('roc_auc', 'mean'),
    Std_ROC_AUC=('roc_auc', 'std'),
    Mean_F1=('f1', 'mean'),
    Mean_Recall=('recall', 'mean'),
    Mean_Precision=('precision', 'mean')
).reset_index().round(5)

save_table(pred_summary, 'objective1_policy_summary')
print("Policy Predictive Performance Summary:")
display(pred_summary)

# Operational summary
oper_summary = run_results_df.groupby('policy').agg(
    Mean_Inference_p50_ms=('inference_p50_ms', 'mean'),
    Mean_Inference_p95_ms=('inference_p95_ms', 'mean'),
    Mean_Inference_p99_ms=('inference_p99_ms', 'mean'),
    Mean_Adaptation_Count=('adaptation_count', 'mean'),
    Mean_Adaptation_Duration_s=('total_adaptation_duration_s', 'mean'),
    Mean_Wall_Time_s=('wall_time_s', 'mean')
).reset_index().round(4)

save_table(oper_summary, 'objective1_operational_summary')
print("Policy Operational Cost Summary:")
display(oper_summary)

  [TABLE SAVED] objective1_policy_summary.csv (4 rows)
Policy Predictive Performance Summary:


,policy,Mean_PR_AUC,Std_PR_AUC,Median_PR_AUC,Mean_ROC_AUC,Std_ROC_AUC,Mean_F1,Mean_Recall,Mean_Precision
0,P0,0.01821,0.0,0.01821,0.45671,0.0,0.00000,0.00000,0.0
1,P1,0.01830,0.0,0.01830,0.46278,0.0,0.00000,0.00000,0.0
2,P2,0.01821,0.0,0.01821,0.45671,0.0,0.00000,0.00000,0.0
3,P3,0.02994,0.0,0.02994,0.52976,0.0,0.03571,0.01923,0.0


  [TABLE SAVED] objective1_operational_summary.csv (4 rows)
Policy Operational Cost Summary:


,policy,Mean_Inference_p50_ms,Mean_Inference_p95_ms,Mean_Inference_p99_ms,Mean_Adaptation_Count,Mean_Adaptation_Duration_s,Mean_Wall_Time_s
0,P0,0.024,0.0405,0.0575,0.0,0.000,22.220
1,P1,0.015,0.0255,0.0330,2.0,1.243,19.895
2,P2,0.016,0.0280,0.0365,0.0,0.000,19.025
3,P3,0.016,0.0465,0.7865,0.0,0.000,21.175


## 16. Visualizations: Trajectories, Comparison Distributions & Cost Frontiers

We generate publication-quality figures:
1. **Rolling PR-AUC Trajectories** across chronological stream progression.
2. **Policy-Level PR-AUC Comparison** showing individual seed data points.
3. **Inference Latency Distributions** ($p_{50}, p_{95}, p_{99}$).
4. **Performance vs. Adaptation Cost Pareto Frontier**.

In [16]:
sns.set_theme(style="whitegrid", font_scale=1.1)

# --- Figure 1: Rolling PR-AUC Trajectories ---
fig1, ax1 = plt.subplots(figsize=(12, 6))
palette = {"P0": "#1f77b4", "P1": "#ff7f0e", "P2": "#2ca02c", "P3": "#d62728"}

for pol in POLICIES:
    pol_runs = [r for r in all_run_results if r['policy'] == pol]
    if pol_runs and 'rolling_trajectory' in pol_runs[0]:
        tx_points = [pt['stream_tx'] for pt in pol_runs[0]['rolling_trajectory']]
        trajectories = []
        for r in pol_runs:
            praucs = [pt['rolling_pr_auc'] for pt in r['rolling_trajectory']]
            trajectories.append(praucs)
        arr = np.array(trajectories)
        mean_traj = np.mean(arr, axis=0)
        sem_traj = stats.sem(arr, axis=0) if len(pol_runs) > 1 else np.zeros_like(mean_traj)
        
        ax1.plot(tx_points, mean_traj, label=f"{pol} (Mean)", color=palette.get(pol, 'black'), lw=2.0)
        ax1.fill_between(tx_points, mean_traj - sem_traj, mean_traj + sem_traj, color=palette.get(pol, 'black'), alpha=0.15)

ax1.set_title("Objective 1: Rolling-Window PR-AUC Trajectories (Temporal Diagnostics)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Streaming Transaction Index ($t$)", fontsize=12)
ax1.set_ylabel("Rolling PR-AUC (Average Precision)", fontsize=12)
ax1.legend(loc="best", frameon=True)
save_figure(fig1, 'objective1_rolling_prauc_trajectories')
plt.close(fig1)

# --- Figure 2: Policy PR-AUC Seed-Level Distribution ---
fig2, ax2 = plt.subplots(figsize=(9, 5))
sns.boxplot(data=run_results_df, x='policy', y='pr_auc', palette=palette, ax=ax2, width=0.4, boxprops=dict(alpha=0.6))
sns.stripplot(data=run_results_df, x='policy', y='pr_auc', color='black', size=7, jitter=0.15, ax=ax2)
ax2.set_title("Objective 1: Full-Stream PR-AUC by Policy Across Random Seeds", fontsize=14, fontweight='bold')
ax2.set_xlabel("Adaptation Policy", fontsize=12)
ax2.set_ylabel("Full-Stream Prequential PR-AUC", fontsize=12)
save_figure(fig2, 'objective1_policy_prauc_comparison')
plt.close(fig2)

# --- Figure 3: Latency Distribution Across Policies ---
lat_melted = run_results_df.melt(
    id_vars=['policy', 'seed'],
    value_vars=['inference_p50_ms', 'inference_p95_ms', 'inference_p99_ms'],
    var_name='percentile',
    value_name='latency_ms'
)
lat_melted['percentile'] = lat_melted['percentile'].map({
    'inference_p50_ms': 'p50',
    'inference_p95_ms': 'p95',
    'inference_p99_ms': 'p99'
})

fig3, ax3 = plt.subplots(figsize=(10, 5))
sns.barplot(data=lat_melted, x='policy', y='latency_ms', hue='percentile', palette='Blues_d', ax=ax3)
ax3.set_title("Objective 1: Inference Latency Percentiles by Adaptation Policy", fontsize=14, fontweight='bold')
ax3.set_xlabel("Adaptation Policy", fontsize=12)
ax3.set_ylabel("Latency (milliseconds)", fontsize=12)
ax3.legend(title="Percentile", frameon=True)
save_figure(fig3, 'objective1_latency_distribution')
plt.close(fig3)

# --- Figure 4: Performance-Cost Pareto Frontier ---
fig4, ax4 = plt.subplots(figsize=(9, 6))
cost_df = run_results_df.groupby('policy').agg(
    Mean_PR_AUC=('pr_auc', 'mean'),
    Sem_PR_AUC=('pr_auc', lambda s: stats.sem(s) if len(s) > 1 else 0.0),
    Mean_Adapt_Time=('total_adaptation_duration_s', 'mean'),
    Sem_Adapt_Time=('total_adaptation_duration_s', lambda s: stats.sem(s) if len(s) > 1 else 0.0)
).reset_index()

for _, row in cost_df.iterrows():
    p = row['policy']
    ax4.errorbar(
        row['Mean_Adapt_Time'], row['Mean_PR_AUC'],
        xerr=row['Sem_Adapt_Time'], yerr=row['Sem_PR_AUC'],
        fmt='o', color=palette.get(p, 'black'), markersize=10,
        label=p, capsize=5, elinewidth=1.5
    )
    ax4.annotate(
        f" {p}", (row['Mean_Adapt_Time'], row['Mean_PR_AUC']),
        fontsize=12, fontweight='bold', verticalalignment='center'
    )

ax4.set_title("Objective 1: Performance–Cost Trade-Off Frontier", fontsize=14, fontweight='bold')
ax4.set_xlabel("Cumulative Adaptation Retraining Time (seconds)", fontsize=12)
ax4.set_ylabel("Full-Stream Prequential PR-AUC", fontsize=12)
save_figure(fig4, 'objective1_adaptation_cost_frontier')
plt.close(fig4)

print("All 4 publication figures rendered and saved successfully.")

  [FIGURE SAVED] objective1_rolling_prauc_trajectories.png
  [FIGURE SAVED] objective1_policy_prauc_comparison.png
  [FIGURE SAVED] objective1_latency_distribution.png
  [FIGURE SAVED] objective1_adaptation_cost_frontier.png
All 4 publication figures rendered and saved successfully.


## 17. Experiment Manifest Serialization & Artifact Packaging

We write a machine-readable experiment manifest (`outputs/checkpoints/objective1_e1_e2_manifest.json`) and bundle all outputs into a self-contained ZIP archive in `kaggle_artifacts/notebook04_objective1_e1_e2_artifacts.zip`.

In [17]:
# Serialize experiment manifest
manifest_data = {
    "experiment_name": "Objective 1 (E1/E2) Adaptation Policy Effectiveness",
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "run_mode": RUN_MODE,
    "dataset": "ieee_cis",
    "warmup_rows": actual_warmup,
    "stream_rows": actual_stream,
    "policies": POLICIES,
    "seeds": TARGET_SEEDS,
    "total_runs_completed": len(run_results_df),
    "frozen_config": FROZEN_CONFIG,
    "tables_generated": [
        "objective1_e1_e2_run_results.csv",
        "objective1_e1_pairwise_statistics.csv",
        "objective1_e2_pairwise_statistics.csv",
        "objective1_policy_summary.csv",
        "objective1_operational_summary.csv"
    ],
    "figures_generated": [
        "objective1_rolling_prauc_trajectories.png",
        "objective1_policy_prauc_comparison.png",
        "objective1_latency_distribution.png",
        "objective1_adaptation_cost_frontier.png"
    ]
}

manifest_path = CHECKPOINT_DIR / 'objective1_e1_e2_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest_data, f, indent=2)
print(f"Experiment manifest serialized to: {manifest_path}")

# Package run artifacts into ZIP archive
def package_notebook04_artifacts() -> Path:
    archive_path = KAGGLE_ARTIFACT_DIR / 'notebook04_objective1_e1_e2_artifacts.zip'
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Include generated tables
        for t_name in manifest_data["tables_generated"]:
            t_file = TABLE_DIR / t_name
            if t_file.exists():
                zipf.write(t_file, arcname=f"tables/{t_name}")
        # Include generated figures
        for f_name in manifest_data["figures_generated"]:
            f_file = FIGURE_DIR / f_name
            if f_file.exists():
                zipf.write(f_file, arcname=f"figures/{f_name}")
        # Include manifest
        if manifest_path.exists():
            zipf.write(manifest_path, arcname=f"checkpoints/{manifest_path.name}")
            
    size_kb = archive_path.stat().st_size / 1024
    print(f"Artifact archive created: {archive_path.name} ({size_kb:.1f} KB)")
    
    # In Kaggle environment, copy to root /kaggle/working/ for instant UI download
    kaggle_working = Path('/kaggle/working')
    if kaggle_working.exists() and kaggle_working.is_dir():
        try:
            shutil.copy2(archive_path, kaggle_working / archive_path.name)
            print(f"Copied archive to Kaggle working root: {kaggle_working / archive_path.name}")
        except Exception as e:
            print(f"Note: Could not copy to /kaggle/working: {e}")
            
    return archive_path

artifact_zip = package_notebook04_artifacts()
print(f"Objective 1 artifact bundle ready at: {artifact_zip}")

Experiment manifest serialized to: C:\Projects\Thesis\outputs\checkpoints\objective1_e1_e2_manifest.json
Artifact archive created: notebook04_objective1_e1_e2_artifacts.zip (508.7 KB)
Copied archive to Kaggle working root: \kaggle\working\notebook04_objective1_e1_e2_artifacts.zip
Objective 1 artifact bundle ready at: C:\Projects\Thesis\kaggle_artifacts\notebook04_objective1_e1_e2_artifacts.zip


## 18. Objective 1 Invariant Verification & Research Summary

We verify that all required tables, figures, and statistical columns were produced and assert that the empirical evidence conforms to the thesis methodology.

In [18]:
print("Running Objective 1 Research Invariant Verification...")

# Invariant 1: Total run accounting
expected_runs = len(POLICIES) * len(TARGET_SEEDS)
assert len(run_results_df) == expected_runs, f"Run count mismatch: expected {expected_runs}, got {len(run_results_df)}"

# Invariant 2: P0 zero window retraining
p0_runs = run_results_df[run_results_df['policy'] == 'P0']
assert (p0_runs['adaptation_count'] == 0).all(), "Invariant Violation: P0 performed window retraining."

# Invariant 3: Positive inference latency percentiles
assert (run_results_df['inference_p50_ms'] > 0).all(), "Invariant Violation: Non-positive p50 latency."
assert (run_results_df['inference_p95_ms'] > 0).all(), "Invariant Violation: Non-positive p95 latency."
assert (run_results_df['inference_p99_ms'] > 0).all(), "Invariant Violation: Non-positive p99 latency."

# Invariant 4: E1 table completeness
assert len(e1_df) == 3, f"E1 comparisons mismatch: expected 3, got {len(e1_df)}"
for col in ['Comparison', 'raw_p_value', 'holm_sidak_p', 'cohens_dz', 'hedges_gz']:
    assert col in e1_df.columns, f"Missing column {col} in E1 table."

# Invariant 5: E2 table completeness
assert len(e2_df) == 6, f"E2 comparisons mismatch: expected 6, got {len(e2_df)}"
for col in ['Comparison', 'raw_p_value', 'holm_sidak_p', 'cohens_dz', 'hedges_gz']:
    assert col in e2_df.columns, f"Missing column {col} in E2 table."

# Invariant 6: Artifact archive exists and is non-empty
assert artifact_zip.exists() and artifact_zip.stat().st_size > 1000, "Artifact ZIP archive is missing or corrupted."

print("[PASS] ALL OBJECTIVE 1 INVARIANTS SATISFIED.")
print("The empirical evidence satisfies the locked Master Source of Truth.")

Running Objective 1 Research Invariant Verification...
[PASS] ALL OBJECTIVE 1 INVARIANTS SATISFIED.
The empirical evidence satisfies the locked Master Source of Truth.
